In [0]:
RAW_PATH = (
    "/Volumes/sfn_dev/landing/raw_files/"
    "bcb/ifdata/report_list/"
)

print(RAW_PATH)

In [0]:
df_raw = spark.read.json(RAW_PATH)

df_raw.printSchema()

In [0]:
display(df_raw)

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_raw
    .select(
        F.explode("value").alias("report"),
        F.col("ingestion_date"),
        F.col("_metadata.file_path").alias("_source_file")
    )
    .select(
        F.col("report.NomeRelatorio").alias("NomeRelatorio"),
        F.col("report.NumeroRelatorio").alias("NumeroRelatorio"),

        F.lit("BCB_IFDATA").alias("_source_system"),
        F.col("_source_file"),
        F.col("ingestion_date").alias("_ingestion_date"),
        F.current_timestamp().alias("_bronze_loaded_at")
    )
)

In [0]:
df_bronze.printSchema()
display(df_bronze)

In [0]:
TABLE_NAME = "sfn_dev.bronze.ifdata_report_list"

print(f"Tabela alvo: {TABLE_NAME}")

In [0]:
if not spark.catalog.tableExists(TABLE_NAME):
    (
        df_bronze
        .limit(0)
        .write
        .format("delta")
        .saveAsTable(TABLE_NAME)
    )

    print("Tabela criada com sucesso.")
else:
    print("Tabela já existe.")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    TABLE_NAME
)

(
    target.alias("target")
    .merge(
        df_bronze.alias("source"),
        """
        target._source_file = source._source_file
        AND target.NumeroRelatorio = source.NumeroRelatorio
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE concluído.")

In [0]:
df_check = spark.table(TABLE_NAME)

display(df_check)

print("Total de registros:", df_check.count())

In [0]:
%sql
DESCRIBE HISTORY sfn_dev.bronze.ifdata_report_list;